<a href="https://colab.research.google.com/github/gustavopierre/flight_delay_project/blob/main/notebook/Flight_Delay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![author](https://github.com/gustavopierre/data_science_portfolio/blob/master/author-gustavopierre.png?raw=true)](https://www.linkedin.com/in/gustavo_pierre) [![](https://img.shields.io/badge/python-3.7+-blue.svg)](https://www.python.org/downloads/release/python-365/) [![contributions welcome](https://img.shields.io/badge/contributions-welcome-brightgreen.svg?style=flat)](https://github.com/gustavopierre/data_science_portfolio/issues)

<p align="center">
  <img src="https://github.com/gustavopierre/flight_delay_project/blob/main/images/pexels-matoga-27982027.jpg?raw=true" alt="Avi?o lvantando v?o no alvorecer" >
  <a href="https://www.pexels.com/pt-br/foto/voo-alvorecer-amanhecer-aurora-27982027/">Foto de Manuel Torres Garcia no Pexels</a>
</p>

# **Predizendo se um V?o Vai Atrasar ou N?o**

## **Resumo**
O Flight Delay Project ? um projeto da sprint de Qualidade de Software, Seguran?a e Sistemas Inteligentes do curso de P?s-gradua??o em Engenharia de Software da Pontif?cia Universidade Cat?lica do Rio de Janeiro. Ele envolve a escolha de um modelo para predizer se um v?o vai ter atraso ou n?o baseado em dados de aeroportos nos EUA disponibilizados no Kaggle.

## **Introdu??o**

## **Inicializando o Ambiente**

In [ ]:
# config don't show warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# library needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC


In [ ]:
# connecting to Google Drive
#from google.colab import drive
#drive.mount('/content/drive')

# **An?lise Explorat?ria dos Dados**

Vamos criar o dataframe e analisa-lo.

In [ ]:
# reading the dataset
path = "https://raw.githubusercontent.com/gustavopierre/flight_delay_project/refs/heads/main/api/machinelearning/data/Airlines.csv"

dt = pd.read_csv(path)

In [ ]:
# showing the shape of the dataset
print("Shape of the dataset:", dt.shape)

In [ ]:
# show the five firsts lines
dt.head()

In [ ]:
# Checking the types of the data
dt.info()

In [ ]:
# Checking the description of the data
dt.describe(include='all')

In [ ]:
# checking the number of null values
# To check the number of null values in each column:
print(dt.isnull().sum())
# To check if there are any null values in the entire DataFrame:
print(dt.isnull().any())
# To check the total number of null values in the DataFrame:
print(dt.isnull().sum().sum())

Flight, Airline, AirportFrom, AirportTo s?o vari?veis categoricas e DayofWeek e Time s?o vari?veis C?clicas. Ambos os tipos n?o s?o adequados para trabalhar em modelos de Machine Learning. O tipo da vari?vel e sua cardinalidade influenciar?o na forma que prepararemos as variaveis.

In [ ]:
# checking the cardinality of the variables
# Cardinality refers to the number of unique values in a column.
variables = ['Flight', 'Airline', 'AirportFrom', 'AirportTo', 'DayOfWeek', 'Time']
for var in variables:
    if dt[var].nunique() < 50:
        cardinality = 'low cardinality'
    else:
        cardinality = 'high cardinality'
    print(f"Vari?vel: {var}, Cardinalidade: {cardinality}, Quantidade de valores ?nicos: {dt[var].nunique()}")

### **Analisando e tratando a vari?vel Flight**

In [ ]:
# Verificando se tem relacao unica entre Flight e 'Airline', 'AirportFrom', 'AirportTo', 'DayOfWeek', 'Time'
# Agrupar pelas colunas desejadas e contar voos ?nicos
flight_counts = dt.groupby(
    ['Airline', 'AirportFrom', 'AirportTo', 'DayOfWeek', 'Time']
)['Flight'].nunique().reset_index(name='UniqueFlightsCount')

# Showing the first few rows of the grouped DataFrame
print(flight_counts.head())

**Flight** ? uma vari?vel que n?o representa uma quantidade, portanto, n?o tem ordem nem escala. N?o ? quantitativa, mas sim qualitativa, possui Cardinalidade Alta (6585 registros ?nicos) e tem criterios proprios em cada empresa a?rea. N?o contem um valor ?nico para uma mesma combina??o das vari?veis 'Airline', 'AirportFrom', 'AirportTo', 'DayOfWeek', 'Time'.  Ser? tratada com Frequency Encoding, mais simples e eficiente. Ser? usada a frequ?ncia relativa ao total, pois o valor ja fica entre 0 e 1.

In [ ]:
# Analyzing and treating the variable Flight
dt["Flight_freq"] = dt['Flight'].map(dt['Flight'].value_counts(normalize=True))
dt[["Flight", "Flight_freq"]].head(10)

### **Analisando e tratando a vari?vel Airline**

**Airline** ?  uma vari?vel categ?rica nominal, portanto, qualitativa. Devido a sua baixa cardinalidade, com apenas 18 valores ?nicos, ser? processada usando One-Hot Encoding, onde ser? criada uma coluna para cada Airline, contendo o valor 0 (falso) ou 1 (verdadeiro).

In [ ]:
# Analyzing and treating the variable Airline
# Encoding the 'Airline' column using one-hot encoding
dt_encoded = pd.get_dummies(dt['Airline'], prefix='Airline')
dt = pd.concat([dt, dt_encoded], axis=1)
dt.head()

In [ ]:
# Checking the types of the data again after encoding
dt.info()

### **Analisando e tratando as vari?veis AirportFrom e AirportTo**

**AirportFrom** e **AirportTo** s?o vari?veis categ?ricas nominais de alta cardinalidade, com 293 valores ?nicos cada uma. Ser? usado *Frequency Encoding* para trat?-las.

In [ ]:
# Analyzing and treating the variables 'AirportFrom' and 'AirportTo'
# Creating frequency columns for 'AirportFrom' and 'AirportTo'
for col in ['AirportFrom', 'AirportTo']:
    freq = dt[col].value_counts(normalize=True)
    dt[col + '_freq'] = dt[col].map(freq)

dt.head()

### **Analisando a vari?vel DayOfWeek**

**DayOfWeek** ? uma vari?vel categ?rica ordinal c?clica. Ser? tranformada em seno e cosseno.

In [ ]:
# Analyzing the variable DayOfWeek
# Creating sine and cosine transformations for 'DayOfWeek'
dt['DayOfWeek_sin'] = np.sin(2 * np.pi * dt['DayOfWeek'] / 7)
dt['DayOfWeek_cos'] = np.cos(2 * np.pi * dt['DayOfWeek'] / 7)
dt.head(10)

### **Analisando a vari?vel Time**

Time ? uma vari?vel categ?rica cont?nua c?clica que refere-se ao tempo em minutos a partir de meia-noite. Ser? tranformada em seno e cosseno.

In [ ]:
# Analyzing the variable Time
# Creating sine and cosine transformations for 'Time'
dt['Time_sin'] = np.sin(2 * np.pi * dt['Time'] / (24 * 60))
dt['Time_cos'] = np.cos(2 * np.pi * dt['Time'] / (24 * 60))

In [ ]:
dt.head()

In [ ]:

# Seleciona as colunas de Length em diante
# Acha a coluna Length
length_col_index = dt.columns.get_loc('Length')

# Seleciona as colunas Length at? o fim
dt_numerical_features = dt.iloc[:, length_col_index:]

# Display the first few rows of the new DataFrame
dt_numerical_features.head()

In [ ]:
dt_numerical_features.info()

Como o dataset tem 539383 registros e essa quantidade nao permitiu acabar de testar os modelos, principalmente o SVM, usaremos somente um subconjunto dele para chegar no modelo de ML. Manteremos a proporcao da coluna target nesta divisao. Nos testes, reduzimos em 50%, depois 20% e, finalmente, chagamos a 10% dos registros, ou seja, 53938 registros.

In [ ]:
seed = 3
# Definindo a vari?vel target
y = dt_numerical_features['Delay']

# Definindo as features (todas as outras colunas)
X = dt_numerical_features.drop('Delay', axis=1)

# Definindo o tamanho do subconjunto para 10% do dataset original
subset_size = 0.10

# Realizando a amostragem estratificada
X_subset, _, y_subset, _ = train_test_split(
    X, y,
    test_size=(1-subset_size),  # Usamos 1 - subset_size para obter o tamanho desejado no primeiro output
    random_state=seed,
    stratify=y # Isso garante que as propor??es de 'Delay' sejam mantidas
)

# Criando um novo DataFrame com o subconjunto
dt_subset = X_subset.copy()
dt_subset['Delay'] = y_subset

print("Shape do dataset original:", dt_numerical_features.shape)
print("Shape do subconjunto:", dt_subset.shape)

print("\nDistribui??o da vari?vel 'Delay' no dataset original:")
display(dt_numerical_features['Delay'].value_counts(normalize=True))

print("\nDistribui??o da vari?vel 'Delay' no subconjunto:")
display(dt_subset['Delay'].value_counts(normalize=True))

Dividindo o subconjunto em treino e teste.

In [ ]:
test_size = 0.2
seed = 3

# Define y como a coluna 'Delay'
y = dt_subset['Delay']

# Define X como todas as colunas menos 'Delay'
X = dt_subset.drop('Delay', axis=1)


X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=test_size, shuffle=True, random_state=seed, stratify=y)

scoring = "accuracy"
num_particoes = 10
kfold = StratifiedKFold(n_splits=num_particoes, shuffle=True, random_state=seed)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

# **Analisando os Modelos**

Analisando KNN, CART, NB e SVM.

In [ ]:
np.random.seed(3)

models = []

models.append(("KNN", KNeighborsClassifier()))
models.append(("CART", DecisionTreeClassifier()))
models.append(("NB", GaussianNB()))
models.append(("SVM", SVC()))

results = []
names = []

for name, model in models:
    cv_results = cross_val_score(model, X_train, y_train, cv=kfold, scoring=scoring)
    results.append(cv_results)
    names.append(name)
    msg = f"{name}: {cv_results.mean()} ({cv_results.std()})"
    print(msg)

fig = plt.figure(figsize=(15,10))
fig.suptitle("Model Comparison")
ax = fig.add_subplot(111)
plt.boxplot(results)
ax.set_xticklabels(names)
plt.show()

Verificando os modelos apos usar padronizacao e normalizacao.

In [ ]:
np.random.seed(3)

pipelines = []
results = []
names = []

knn = ("KNN", KNeighborsClassifier())
cart = ("CART", DecisionTreeClassifier())
naive_bayes = ("NB", GaussianNB())
svm = ("SVM", SVC())

standard_scaler = ("StandardScaler", StandardScaler())
min_max_scaler = ("MinMaxScaler", MinMaxScaler())

# dataset original
pipelines.append(("KNN-orig", Pipeline([knn])))
pipelines.append(("CART-orig", Pipeline([cart])))
pipelines.append(("NB-orig", Pipeline([naive_bayes])))
pipelines.append(("SVM-orig", Pipeline([svm])))

# dataset padronizado
pipelines.append(("KNN-std", Pipeline([standard_scaler, knn])))
pipelines.append(("CART-std", Pipeline([standard_scaler, cart])))
pipelines.append(("NB-std", Pipeline([standard_scaler, naive_bayes])))
pipelines.append(("SVM-std", Pipeline([standard_scaler, svm])))

# dataset normalizado
pipelines.append(("KNN-norm", Pipeline([min_max_scaler, knn])))
pipelines.append(("CART-norm", Pipeline([min_max_scaler, cart])))
pipelines.append(("NB-norm", Pipeline([min_max_scaler, naive_bayes])))
pipelines.append(("SVM-norm", Pipeline([min_max_scaler, svm])))

# executando os pipelines
for name, model in pipelines:
    cv_results = cross_val_score(model, X_train, y_train, cv=kfold, scoring=scoring)
    results.append(cv_results)
    names.append(name)
    msg = f"{name}: {cv_results.mean()} ({cv_results.std()})"
    print(msg)

# boxplot de compara??o dos modelos
fig = plt.figure(figsize=(25, 6))
fig.suptitle("Compara??o dos Modelos - Dataset orginal, padronizado e normalizado")
ax = fig.add_subplot(111)
plt.boxplot(results)
ax.set_xticklabels(names, rotation=90)
plt.show()

O SVM, seguido do KNN, se mostraram com melhor resultados. Vamos tuna-los e descobrir os melhores parametros para isso. Comecaremos pelo KNN por levar um tempo menor de execucao.

## **KNN**

In [ ]:
# Tuning do KNN
np.random.seed(3)

pipelines = []

knn = ("KNN", KNeighborsClassifier())
standard_scaler = ("StandardScaler", StandardScaler())
min_max_scaler = ("MinMaxScaler", MinMaxScaler())

pipelines.append(("KNN-orig", Pipeline([knn])))
pipelines.append(("KNN-std", Pipeline(steps=[standard_scaler, knn])))
pipelines.append(("KNN-norm", Pipeline(steps=[min_max_scaler, knn])))

param_grid = {
    'KNN__n_neighbors': [1,3,5,6,9,11,13,15,17,19,21],
    'KNN__metric': ['euclidean', 'manhattan', 'minkowski'],
}

for name, model in pipelines:
    grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring=scoring, cv=kfold)
    grid.fit(X_train, y_train)
    print(f"Sem tratamento de missings: {name} - Melhor acur?cia: {grid.best_score_} - Melhor par?metro: {grid.best_params_}")

In [ ]:
# Avalia??o do modelo com o conjunto de testes

# Prepara??o do modelo
scaler = StandardScaler().fit(X_train) # ajuste do scaler com o conjunto de treino
rescaledX = scaler.transform(X_train) # aplica??o da padroniza??o no conjunto de treino
modelKNN = KNeighborsClassifier(metric='manhattan', n_neighbors=21)
modelKNN.fit(rescaledX, y_train)

# Estimativa da acur?cia no conjunto de teste
rescaledTestX = scaler.transform(X_test) # aplica??o da padroniza??o no conjunto de teste
predictions = modelKNN.predict(rescaledTestX)
print(accuracy_score(y_test, predictions))

In [ ]:
# Prepara??o do modelo com TODO o dataset
scaler = StandardScaler().fit(X) # ajuste do scaler com TODO o dataset
rescaledX = scaler.transform(X) # aplica??o da padroniza??o com TODO o dataset
modelKNN.fit(rescaledX, y)
print("Modelo KNN treinado com TODO o dataset.")

## **SVM**

Infelizmente a celula abaixo tentando o Tunning do SVM demorou 18h e, portanto, foi interrompido. Portanto, serao usado o SVM com os dados normalizados e padronizados, sem Tunning. Permanece abaixo a celula para Tunning do SVM comentado para se o leitor se interessar em tentar executa-la.

In [ ]:
# Tuning do SVM
""" np.random.seed(3)

pipelines = []

svm = ("SVM", SVC())
standard_scaler = ("StandardScaler", StandardScaler())
min_max_scaler = ("MinMaxScaler", MinMaxScaler())

# pipelines.append(("SVM-orig", Pipeline([svm])))
# pipelines.append(("SVM-std", Pipeline(steps=[standard_scaler, svm])))
pipelines.append(("SVM-norm", Pipeline(steps=[min_max_scaler, svm])))

param_grid = {
    'SVM__C': [0.1, 1, 10, 100],
    'SVM__kernel': ['linear', 'rbf', 'poly'],
#    'SVM__gamma': ['scale', 'auto']  # S? ? usado para alguns kernels
}

for name, model in pipelines:
    grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring=scoring, cv=kfold)
    grid.fit(X_train, y_train)
    print(f"Sem tratamento de missings: {name} - Melhor acur?cia: {grid.best_score_} - Melhor par?metro: {grid.best_params_}") """


In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
# Avalia??o do modelo com o conjunto de testes

# Prepara??o do modelo
# Pipeline: StandardScaler -> MinMaxScaler -> SVM
pipelineModelSVM = Pipeline([
    ('standard', StandardScaler()),
    ('minmax', MinMaxScaler()),
    ('svm', SVC())  # kernel e gamma padr?o
])

pipelineModelSVM.fit(X_train, y_train)
predictions = pipelineModelSVM.predict(X_test)
print("Acur?cia com StandardScaler + MinMaxScaler + SVM:", accuracy_score(y_test, predictions))



In [ ]:
X.shape, y.shape

In [ ]:
# Prepara??o do modelo com TODO o dataset

# Pipeline: StandardScaler -> MinMaxScaler -> SVM
pipeline_full_ModelSVM = Pipeline([
    ('standard', StandardScaler()),
    ('minmax', MinMaxScaler()),
    ('svm', SVC())  # usa par?metros padr?o
])

# Treina o pipeline com TODO o dataset
pipeline_full_ModelSVM.fit(X, y)
print("Modelo SVM treinado com TODO o dataset usando StandardScaler + MinMaxScaler.")

# **Preparando o Pipeline**

Nas fases anteriores, fizemos uma An?lise Explorat?ria dos Dados (AED) e chegamos a um modelo com melhor acur?cia. Agora vamos criar o pipeline para gerar o arquivo que usarmos no sistema que ser? desenvolvido. Partiremos do in?cio, lendo o dataset e reduzindo-o. O processamento das vari?vies ser? realizado utilizando o recursos da biblioteca scikit-learn.

In [ ]:
# carregando o dataset novamente
path = "https://raw.githubusercontent.com/gustavopierre/flight_delay_project/refs/heads/main/api/machinelearning/data/Airlines.csv"

dt = pd.read_csv(path)
dt.head()

Vamos reduzir o dataset para 10% do original devido ao tempo de processamento, como j? mostrado na AED.

In [ ]:
seed = 3
# Definindo a vari?vel target
y = dt['Delay']

# Definindo as features (todas as outras colunas)
X = dt.drop('Delay', axis=1)

# Definindo o tamanho do subconjunto (por exemplo, 20%)
subset_size = 0.10

# Realizando a amostragem estratificada
X_subset, _, y_subset, _ = train_test_split(
    X, y,
    test_size=(1-subset_size),  # Usamos 1 - subset_size para obter o tamanho desejado no primeiro output
    random_state=seed,
    stratify=y # Isso garante que as propor??es de 'Delay' sejam mantidas
)

# Criando um novo DataFrame com o subconjunto
dt_subset = X_subset.copy()
dt_subset['Delay'] = y_subset

print("Shape do dataset original:", dt.shape)
print("Shape do subconjunto:", dt_subset.shape)

print("\nDistribui??o da vari?vel 'Delay' no dataset original:")
display(dt['Delay'].value_counts(normalize=True))

print("\nDistribui??o da vari?vel 'Delay' no subconjunto:")
display(dt_subset['Delay'].value_counts(normalize=True))

Para criar um pipeline que trata diferentes tipos de vari?veis de forma adequada, utilizaremos o `ColumnTransformer` do scikit-learn. Este transformador permite aplicar diferentes transforma??es a diferentes subconjuntos de colunas.

As transforma??es ser?o as seguintes:
- **Vari?veis de Alta Cardinalidade e Nominal**: 'Flight', 'AirportFrom', 'AirportTo' ser?o tratadas com Frequency Encoding.
- **Vari?vel de Baixa Cardinalidade e Nominal**: 'Airline' ser? tratada com One-Hot Encoding.
- **Vari?veis C?clicas**: 'DayOfWeek' e 'Time' ser?o transformadas em suas componentes seno e cosseno.
- **Vari?vel Num?rica Cont?nua**: 'Length' ser? padronizada usando `StandardScaler`.

As vari?veis transformadas (freq encoding, one-hot encoding, seno/cosseno) tamb?m precisar?o ser padronizadas, o que ser? feito ap?s o `ColumnTransformer` com outro `StandardScaler`.

Finalmente, o modelo `KNeighborsClassifier` com os melhores par?metros encontrados (`n_neighbors=21`, `metric='manhattan'`) ser? aplicado para gerar o pipeline model_pipeline_KNN e, para gerar o model_pipeline_SVM sera aplicado o modelo `SVC` sem os melhores parametros, devido ao tempo de execucao, conforme descrito em topico anterior.

In [ ]:
# Define as colunas para cada tipo de transforma??o
high_cardinality_cols = ['Flight', 'AirportFrom', 'AirportTo']
low_cardinality_cols = ['Airline']
cyclic_cols = ['DayOfWeek', 'Time']
numerical_cols = ['Length']
target_col = 'Delay'

# Cria os transformadores personalizados
def frequency_encoding(X):
    X_copy = X.copy()
    for col in high_cardinality_cols:
        freq = X_copy[col].value_counts(normalize=True)
        X_copy[col] = X_copy[col].map(freq)
    return X_copy[high_cardinality_cols] # Retorna apenas as colunas transformadas

def cyclic_transformer(X):
    X_copy = X.copy()
    for col in cyclic_cols:
        if col == 'DayOfWeek':
            X_copy[col + '_sin'] = np.sin(2 * np.pi * X_copy[col] / 7)
            X_copy[col + '_cos'] = np.cos(2 * np.pi * X_copy[col] / 7)
        elif col == 'Time':
            X_copy[col + '_sin'] = np.sin(2 * np.pi * X_copy[col] / (24 * 60))
            X_copy[col + '_cos'] = np.cos(2 * np.pi * X_copy[col] / (24 * 60))
    return X_copy[[col + '_sin' for col in cyclic_cols] + [col + '_cos' for col in cyclic_cols]] # Retorna apenas as colunas transformadas


# Cria os pipelines para cada tipo de feature
preprocessor = ColumnTransformer(
    transformers=[
        ('freq', FunctionTransformer(func=frequency_encoding), high_cardinality_cols),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), low_cardinality_cols),
        ('cyclic', FunctionTransformer(func=cyclic_transformer), cyclic_cols),
        ('scaler_num', StandardScaler(), numerical_cols) # Padroniza 'Length'
    ],
    remainder='passthrough' # Mant?m as outras colunas (neste caso, 'id' e 'Delay' - que ser?o removidas antes de passar para o pipeline)
)



# Separa as features (X) e o target (y) do subset
X_subset_for_pipeline = dt_subset.drop(target_col, axis=1)
y_subset_for_pipeline = dt_subset[target_col]

# Remove a coluna 'id'
X_subset_for_pipeline = X_subset_for_pipeline.drop(['id'], axis=1)


# Divide o subconjunto em treino e teste para o pipeline
X_train_pipe, X_test_pipe, y_train_pipe, y_test_pipe = train_test_split(
    X_subset_for_pipeline, y_subset_for_pipeline,
    test_size=test_size,
    shuffle=True,
    random_state=seed,
    stratify=y_subset_for_pipeline
)



In [ ]:
X_subset.shape, y_subset.shape, X_subset_for_pipeline.shape, y_subset_for_pipeline.shape, X_train_pipe.shape, X_test_pipe.shape, y_train_pipe.shape, y_test_pipe.shape

In [ ]:
# Cria os pipelines completos
model_pipeline_KNN_scaler = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()), # Padroniza todas as features ap?s o preprocessor
    ('knn', KNeighborsClassifier(n_neighbors=21, metric='manhattan'))
])

# Treina o pipeline
model_pipeline_KNN_scaler.fit(X_train_pipe, y_train_pipe)

# Avalia o pipeline no conjunto de teste
accuracy = model_pipeline_KNN_scaler.score(X_test_pipe, y_test_pipe)
print(f"Acur?cia do pipeline KNN scaler no conjunto de teste: {accuracy}")

In [ ]:
# Cria os pipelines completos
model_pipeline_KNN = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()), # Padroniza todas as features ap?s o preprocessor
    ('minmax', MinMaxScaler()), # Normaliza todas as features ap?s o preprocessor
    ('knn', KNeighborsClassifier(n_neighbors=21, metric='manhattan'))
])

# Treina o pipeline
model_pipeline_KNN.fit(X_train_pipe, y_train_pipe)

# Avalia o pipeline no conjunto de teste
accuracy = model_pipeline_KNN.score(X_test_pipe, y_test_pipe)
print(f"Acur?cia do pipeline KNN no conjunto de teste: {accuracy}")

In [ ]:
# Cria os pipelines completos
model_pipeline_SVM = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()), # Padroniza todas as features ap?s o preprocessor
    ('minmax', MinMaxScaler()), # Normaliza todas as features ap?s o preprocessor
    ('svm', SVC()) # Usa par?metros padr?o no SVM
])

# Treina o pipeline
model_pipeline_SVM.fit(X_train_pipe, y_train_pipe)

# Avalia o pipeline no conjunto de teste
accuracy = model_pipeline_SVM.score(X_test_pipe, y_test_pipe)
print(f"Acur?cia do pipeline SVM no conjunto de teste: {accuracy}")

O pipeline foi criado e treinado com sucesso! Agora ele pode ser usado para fazer predi??es em novos dados, garantindo que as mesmas transforma??es aplicadas aos dados de treinamento sejam aplicadas aos novos dados.

Para demonstrar como usar o pipeline para fazer predi??es em novos dados, vou criar um pequeno exemplo de dados de teste.

In [ ]:
# Exemplo de novos dados para predi??o
new_data = pd.DataFrame({
    'Airline': ['CO', 'US', 'AA'],
    'Flight': [269, 1558, 2400],
    'AirportFrom': ['SFO', 'PHX', 'LAX'],
    'AirportTo': ['IAH', 'CLT', 'DFW'],
    'DayOfWeek': [3, 3, 3],
    'Time': [15, 15, 20],
    'Length': [205, 222, 165],
    'id': [1, 2, 3] # 'id' will be dropped by the pipeline
})

# Remove a coluna 'id' antes de passar para o pipeline para predi??o
new_data_for_prediction = new_data.drop('id', axis=1)

# Faz a predi??o usando o pipeline treinado
predictions_pipeline_KNN_scaler = model_pipeline_KNN_scaler.predict(new_data_for_prediction)
predictions_pipeline_KNN = model_pipeline_KNN.predict(new_data_for_prediction)
predictions_pipeline_SVM = model_pipeline_SVM.predict(new_data_for_prediction)

print("\nPredi??es para os novos dados:")
print(f"KNN Scaler: {predictions_pipeline_KNN_scaler}\nKNN: {predictions_pipeline_KNN}\nSVM: {predictions_pipeline_SVM}")

## Gerando Arquivos para o Back-End

In [ ]:
import pickle


# Save the trained models to files
with open('model_pipeline_KNN_scaler.pkl', 'wb') as file:
    pickle.dump(model_pipeline_KNN_scaler, file)
with open('model_pipeline_KNN.pkl', 'wb') as file:
    pickle.dump(model_pipeline_KNN, file)
with open('model_pipeline_SVM.pkl', 'wb') as file:
    pickle.dump(model_pipeline_SVM, file)

print("Models saved successfully.")

Testando os arquivos pickle.

In [ ]:
# Exemplo de novos dados para predi??o (certifique-se de que as colunas estejam na mesma ordem que os dados de treino, exceto 'id')
new_data_to_predict = pd.DataFrame({
    'Airline': ['CO', 'US', 'AA'],
    'Flight': [269, 1558, 2400],
    'AirportFrom': ['SFO', 'PHX', 'LAX'],
    'AirportTo': ['IAH', 'CLT', 'DFW'],
    'DayOfWeek': [3, 3, 3],
    'Time': [15, 15, 20],
    'Length': [205, 222, 165],
    # 'id': [1, 2, 3] # 'id' will be dropped by the pipeline
})
# Carrega o pipeline do arquivo
for filename in ['model_pipeline_KNN_scaler.pkl', 'model_pipeline_KNN.pkl', 'model_pipeline_SVM.pkl']:
    with open(filename, 'rb') as file:
        loaded_pipeline = pickle.load(file)

    # Faz a predi??o usando o pipeline carregado
    predictions_loaded_pipeline = loaded_pipeline.predict(new_data_to_predict)
    print(f"\nPredi??es usando o pipeline carregado ({filename}):")
    print(predictions_loaded_pipeline)
